# Imports 

In [1]:
#IMPORTING stuff including my online repo with useful tools for processing chess files (PGN)

import sys
import os, sys
if os.path.exists("/tmp/ludus_scacchi"):
    print("Repository already exists, skipping clone.")
    sys.path.append("/tmp/ludus_scacchi")
else:
    !git clone https://github.com/paolomostardi/ludus_scacchi.git /tmp/ludus_scacchi
    !pip install chess
    sys.path.append("/tmp/ludus_scacchi")
from Backend.data_pipeline import gm_pipeline
from Backend.data_pipeline import from_PGN_generate_bitboards as gen
import pandas as pd
import numpy as np
!pip install chess mlflow
import mlflow
from pathlib import Path
Path('/kaggle/working/Backend/model_evaluation').mkdir(parents=True, exist_ok=True)
from datetime import datetime
from Backend.model_evaluation.pipeline import model_evaluation as evals

Cloning into '/kaggle/working/ludus_scacchi'...
remote: Enumerating objects: 1257, done.
remote: Total 1257 (delta 0), reused 0 (delta 0), pack-reused 1257 (from 1)
Receiving objects: 100% (1257/1257), 160.62 MiB | 32.42 MiB/s, done.
Resolving deltas: 100% (777/777), done.
Updating files: 100% (101/101), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 50.8 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=e13ed426d72be8f1f9639fe394f6bc96ef001a049c29e78b1765bd36c4f25626
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2026-03-28 11:41:40.083209: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774698100.344637      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774698100.423172      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774698101.062916      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774698101.062958      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774698101.062961      55 computation_placer.cc:177] computation placer alr

# Setting the enviorement 

In [ ]:
# Configure the same remote MLflow server used by fit-model.ipynb.
from kaggle_secrets import UserSecretsClient
import os
from pathlib import Path
user_secrets = UserSecretsClient()
mlflow_ip = user_secrets.get_secret("IP_MLFLOW")
mlflow_username = user_secrets.get_secret("USERNAME_MLFLOW")
mlflow_password = user_secrets.get_secret("PASSWORD_MLFLOW")
os.environ["MLFLOW_TRACKING_URI"] = f"http://{mlflow_ip}"
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "true"
os.environ["MLFLOW_TRACKING_USERNAME"] = mlflow_username
os.environ["MLFLOW_TRACKING_PASSWORD"] = mlflow_password
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
EXPERIMENT_NAME = "ludus_cnn"
mlflow.set_experiment(EXPERIMENT_NAME)
print("MLflow tracking configured; server address omitted from notebook output.")
print("MLflow experiment:", EXPERIMENT_NAME)

In [ ]:
from pathlib import Path
FIT_MODEL_INPUT = Path("/kaggle/input/notebooks/paolomostardi/fit-model")
INPUT_RUN_ID_FILE = FIT_MODEL_INPUT / "mlflow_ids.txt"
if not INPUT_RUN_ID_FILE.exists():
    raise FileNotFoundError(f"Missing {INPUT_RUN_ID_FILE}. Make sure the fit notebook exported its MLflow IDs.")
run_metadata = dict(line.split("=", 1) for line in INPUT_RUN_ID_FILE.read_text().splitlines() if "=" in line)
required_ids = {"run_id", "experiment_id"}
missing_ids = required_ids - run_metadata.keys()
if missing_ids:
    raise ValueError(f"Missing MLflow IDs in {INPUT_RUN_ID_FILE}: {sorted(missing_ids)}")
MLFLOW_RUN_ID = run_metadata["run_id"]
EXPERIMENT_ID = run_metadata["experiment_id"]
print(f"Using training run {MLFLOW_RUN_ID} in experiment {EXPERIMENT_ID}")

In [ ]:
import json
from Backend.model_architecture.implementation.experimental.model_testing_mlflow import model_testing_mlflow
from Backend.model_architecture.implementation.experimental.source_resnet import source_resnet

config_path = FIT_MODEL_INPUT / "model_config.json"
if config_path.is_file():
    model_config = json.loads(config_path.read_text(encoding="utf-8"))
    if model_config.get("schema_version") != 1:
        raise ValueError("Unsupported model configuration schema.")
    if (model_config.get("run_id") != MLFLOW_RUN_ID
            or model_config.get("experiment_id") != EXPERIMENT_ID):
        raise ValueError("Model configuration belongs to a different training run.")
else:
    # Older fit outputs have these settings in MLflow but no configuration file.
    # Read the recorded values so existing models can be evaluated without retraining.
    recorded_run = mlflow.get_run(MLFLOW_RUN_ID)
    if recorded_run.info.experiment_id != EXPERIMENT_ID:
        raise ValueError("Training run belongs to a different experiment.")
    params = recorded_run.data.params
    required = {"model_factory", "conv_filters", "upsample_factor", "model_name"}
    missing = required - params.keys()
    if missing:
        raise ValueError(f"Training run lacks model configuration: {sorted(missing)}")
    model_config = {
        "factory": params["model_factory"],
        "parameters": {
            "conv_filters": int(params["conv_filters"]),
            "upsample_factor": int(params["upsample_factor"]),
        },
        "artifact_file": params["model_name"] + ".keras",
    }
    print("Using model configuration recorded in MLflow for this older fit output.")

# Explicitly supported factories; do not import arbitrary code from metadata.
model_factories = {
    "model_testing_mlflow": (model_testing_mlflow, {"conv_filters", "upsample_factor"}),
    "source_resnet": (source_resnet, {"conv_filters", "residual_blocks"}),
}
factory_name = model_config.get("factory")
if factory_name not in model_factories:
    raise ValueError(f"Unsupported model factory: {factory_name!r}. Add its factory before evaluation.")
parameters = model_config.get("parameters")
factory, required_parameters = model_factories[factory_name]
if not isinstance(parameters, dict) or set(parameters) != required_parameters:
    raise ValueError(f"Model configuration must supply {sorted(required_parameters)}.")
if any(type(value) is not int or value <= 0 for value in parameters.values()):
    raise ValueError("Model parameters must be positive integers.")
artifact_file = model_config.get("artifact_file")
if (not isinstance(artifact_file, str) or not artifact_file.endswith(".keras")
        or "/" in artifact_file or "\\" in artifact_file):
    raise ValueError("Model artifact must be a .keras filename within the fit output.")
model_path = FIT_MODEL_INPUT / artifact_file
if not model_path.is_file():
    raise FileNotFoundError(f"Training model artifact is missing: {model_path}")

# Rebuild from the saved settings and load weights to avoid cross-version
# deserialization of the full Keras model configuration.
model = factory(**parameters)
model.load_weights(str(model_path))
print(f"Loaded {factory_name} with recorded parameters {parameters}.")


## settings ids from text

In [ ]:
training_run = mlflow.get_run(MLFLOW_RUN_ID)
if training_run.info.experiment_id != EXPERIMENT_ID:
    raise ValueError(f"Run {MLFLOW_RUN_ID} belongs to experiment {training_run.info.experiment_id}, not {EXPERIMENT_ID}.")
print(f"Found training run {MLFLOW_RUN_ID} in the configured MLflow server.")

# eval loop

In [ ]:
import os
pipeline_mode = os.environ.get("LUDUS_PIPELINE_MODE", "training")
if pipeline_mode not in {"training", "validation"}:
    raise ValueError(f"Unknown pipeline mode: {pipeline_mode}")
max_evaluation_rows = None
if pipeline_mode == "validation":
    max_evaluation_rows = int(os.environ.get("LUDUS_VALIDATION_EVAL_ROWS", "1000"))
    if max_evaluation_rows < 1:
        raise ValueError("Validation evaluation row limit must be positive.")

from Backend.data_pipeline import gm_pipeline
from Backend.data_pipeline import from_PGN_generate_bitboards as gen
from Backend.model_evaluation import create_dataset_for_simple_positions as create_simple

from pathlib import Path
input_root = Path('/kaggle/input')
run_config = json.loads(os.environ["LUDUS_RUN_CONFIG"]) if "LUDUS_RUN_CONFIG" in os.environ else {}
evaluation_dataset_name = run_config.get("evaluation_dataset", "paolomostardi/eval-training-2013-chunk-9").split("/")[-1]
evaluation_roots = [p for p in input_root.rglob(evaluation_dataset_name) if p.is_dir()]
if not evaluation_roots:
    mounted_inputs = sorted(p.name for p in input_root.iterdir())
    raise FileNotFoundError(
        f"Dataset 'eval-training-2013-chunk-9' is not mounted under {input_root}. Mounted inputs: {mounted_inputs}."
    )
if len(evaluation_roots) > 1:
    raise RuntimeError(f"Found multiple evaluation dataset roots: {evaluation_roots}")
evaluation_data_dir = evaluation_roots[0] / 'my_processed_chunks' / '2013-01'
x_path = evaluation_data_dir / 'chunk_6_x.npy'
y_path = evaluation_data_dir / 'chunk_6_y.npy'
if not x_path.is_file() or not y_path.is_file():
    raise FileNotFoundError(f"Expected evaluation arrays are missing under {evaluation_data_dir}")
x = np.load(x_path, mmap_mode='r')
y = np.load(y_path, mmap_mode='r')
if len(x) != len(y) or len(x) == 0:
    raise ValueError("Evaluation inputs and targets must have equal nonzero lengths.")
if max_evaluation_rows is not None:
    x = x[:max_evaluation_rows]
    y = y[:max_evaluation_rows]

from Backend.model_evaluation.simple_positions import load_catalog, prepare, score_predictions
simple_catalog = load_catalog()
x_simple, y_simple = prepare(simple_catalog)

if y.ndim == 4 and y.shape[1:] == (2, 8, 8):
    y_eval = np.array([target[0].flat for target in y])
else:
    y_eval = y

evaluation_datasets = [
    ('simple_v2', x_simple, y_simple),
    ('2013_01', x, y_eval),
]

print('Prepared evaluation datasets:')
for dataset_name, dataset_x, dataset_y in evaluation_datasets:
    print(dataset_name, dataset_x.shape, dataset_y.shape)

In [ ]:
from pathlib import Path
import json
import time
import mlflow
from Backend.training_pipeline.validation import validate_arrays, validate_metrics
from Backend.training_pipeline.runtime import runtime_metadata, save_worker_requirements
runtime = runtime_metadata()
save_worker_requirements(runtime, Path("/kaggle/working/evaluation_requirements.txt"))
Path("/kaggle/working/evaluation_runtime.json").write_text(json.dumps(runtime, indent=2))
from keras.metrics import TopKCategoricalAccuracy

model.compile(
    optimizer=model.optimizer,
    loss=model.loss,
    metrics=["accuracy", TopKCategoricalAccuracy(k=3, name="top3_accuracy")],
)

evaluation_results = {
    "pipeline_mode": pipeline_mode,
    "max_evaluation_rows": max_evaluation_rows,
    "run_id": MLFLOW_RUN_ID,
    "experiment_id": EXPERIMENT_ID,
    "mlflow_logged": False,
    "datasets": {},
}
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

# Save the local report before remote logging, so computed metrics remain available
# even if the tracking server rejects a metric write.
for test_name, dataset_x, dataset_y in evaluation_datasets:
    validate_arrays(dataset_x, dataset_y)
    if test_name == 'simple_v2':
        simple_diagnostics = score_predictions(simple_catalog, model.predict(dataset_x, batch_size=128, verbose=0))
        metrics_dict = simple_diagnostics['metrics']
        simple_report_path = Path('/kaggle/working/simple_positions_report.json')
        simple_report_path.write_text(json.dumps(simple_diagnostics, indent=2) + '\n', encoding='utf-8')
        evaluation_results['simple_dataset'] = simple_diagnostics['dataset']
        evaluation_results['simple_diagnostics_file'] = simple_report_path.name
    else:
        metrics_dict = model.evaluate(
            dataset_x, dataset_y, batch_size=128, verbose=0, return_dict=True
        )
    validate_metrics(metrics_dict)
    evaluation_results["datasets"][test_name] = {
        "accuracy": float(metrics_dict["accuracy"]),
        "loss": float(metrics_dict["loss"]),
        "top3_accuracy": float(metrics_dict["top3_accuracy"]),
        "sample_count": int(len(dataset_x)),
    }

report_path = Path("/kaggle/working/evaluation_report.json")
report_path.write_text(json.dumps(evaluation_results, indent=2) + "\n", encoding="utf-8")

# Log metrics individually with distinct timestamps and dataset steps. This avoids
# the duplicate-row conflict caused by batched metric writes on the tracking server.
with mlflow.start_run(run_id=MLFLOW_RUN_ID):
    mlflow.set_tags({f'simple_v2_{key}': str(value) for key, value in simple_diagnostics['dataset'].items()})
    mlflow.log_params({f"evaluation_runtime_{name}": runtime["packages"].get(name, "unknown")
                       for name in ("tensorflow", "keras", "numpy", "mlflow", "chess")})
    mlflow.log_param("evaluation_runtime_python", runtime["python"])
    mlflow.log_param("evaluation_repository_revision", runtime["repository_revision"])
    base_timestamp = int(time.time() * 1000)
    metric_offset = 0
    for dataset_step, (test_name, metrics) in enumerate(evaluation_results["datasets"].items(), start=1):
        for metric_name in ("accuracy", "loss", "top3_accuracy", "sample_count"):
            key = f"{metric_name}_{test_name}"
            value = metrics[metric_name]
            mlflow.log_metric(
                key,
                value,
                timestamp=base_timestamp + metric_offset,
                step=dataset_step,
                synchronous=True,
            )
            metric_offset += 1
    evaluation_results["mlflow_logged"] = True
    report_path.write_text(json.dumps(evaluation_results, indent=2) + "\n", encoding="utf-8")
    mlflow.log_artifact(str(report_path), artifact_path="evaluation")
    mlflow.log_artifact(str(simple_report_path), artifact_path="evaluation")
print(json.dumps(evaluation_results, indent=2))

## Running the evaluation

1. Run the fit notebook and attach its output to this notebook. The fit output includes the trained `.keras` model, `mlflow_ids.txt`, and `model_config.json`. Evaluation reconstructs the model from that configuration; older outputs fall back to the settings recorded in their MLflow run. Unsupported factories or missing settings stop evaluation with a clear error.
2. Execute the setup and data cells in order. They connect to the same remote MLflow server, load the model, and prepare the evaluation chunk and simple positions.
3. Run the evaluation cell. It records named loss and accuracy metrics under the training run ID.
4. Review the metrics in MLflow together with the model artifact and the training configuration.

Update the input dataset paths when evaluating a different month or chunk.
